# New courier's delivery time against a 72-hour promise

> **Before you run:** choose **Copy to Drive** at the top of this page (or **File → Save a copy in Drive**). Your copy opens in a new tab. In your copy, choose **Runtime → Run all**. Your copy is yours, so Colab runs it without a warning, and it keeps your changes.

**Bayesian Estimation of the Population Mean** · a worked example from the StatNex teaching guide

**The story.** An online shop tried a new courier for 20 orders and timed each one from dispatch to the customer's door. The courier promises "72 hours on average". The shop's own customers, though, complain about single late parcels, not about averages.

**The question.** How likely is it that the courier's true average is under 72 hours, and how long might one customer wait?

If you run this page without a copy, Colab first warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## Before the analysis

This analysis needs `pymc` and `arviz`. This cell installs them. If Colab asks you to restart the session, restart it and choose **Run all** again.

In [ ]:
%pip install -q "pymc>=6.3" "arviz>=1.3"

## The table

This cell saves the table `14_bayesian/bayes_mean.csv` (20 rows) as `data.csv`, the file the code reads.

In [ ]:
with open("data.csv", "w", encoding="utf-8") as file:
    file.write("""\
Order no.,Delivery time (hours)
1.0,71.0
2.0,63.0
3.0,64.0
4.0,42.0
5.0,89.0
6.0,82.0
7.0,65.0
8.0,78.0
9.0,72.0
10.0,63.0
11.0,80.0
12.0,66.0
13.0,65.0
14.0,60.0
15.0,74.0
16.0,68.0
17.0,75.0
18.0,62.0
19.0,70.0
20.0,59.0
""")

## Libraries

In [ ]:
import arviz as az
import numpy as np
import pandas as pd
import pymc as pm
from scipy import stats

pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:.6g}".format)

## The data

In [ ]:
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
variables = ["Delivery time (hours)"]
credible = 95 / 100
low, high = (1 - credible) / 2, (1 + credible) / 2
comparison = 72


def observations(name):
    values = pd.to_numeric(data[name], errors="coerce").dropna().to_numpy(float)
    return values

## Posterior of the mean: PyMC

In [ ]:
# The model: the values are independent draws from a normal distribution with mean mu
# and standard deviation sigma.
# The prior is the non-informative p(mu, sigma^2) ∝ 1/sigma^2: flat for mu and for
# log(sigma). Its credible interval for mu is the t-based confidence interval.
# PyMC samples the posterior by MCMC (NUTS). One new value is drawn for each draw of
# (mu, sigma): that is the posterior predictive distribution, which carries the spread
# of single values as well as the uncertainty about mu.
stats_rows, posterior_rows, chance_rows = {}, [], []
predictive = np.random.default_rng(2026)
for name in variables:
    values = observations(name)
    with pm.Model():
        mu = pm.Flat("mu")
        log_sigma = pm.Flat("log_sigma")  # flat in log(sigma): p(sigma^2) ∝ 1/sigma^2
        sigma = pm.Deterministic("sigma", pm.math.exp(log_sigma))
        pm.Normal("y", mu=mu, sigma=sigma, observed=values)
        trace = pm.sample(draws=25_000, tune=2_000, chains=4, random_seed=2026, progressbar=False,
                          initvals={"mu": values.mean(), "log_sigma": np.log(values.std(ddof=1))})
    print(f"\n{name}")
    print(az.summary(trace, var_names=["mu", "sigma"], kind="stats", ci_prob=credible, ci_kind="eti"))
    mu_draws = trace.posterior["mu"].to_numpy().ravel()
    sigma_draws = trace.posterior["sigma"].to_numpy().ravel()
    new = predictive.normal(mu_draws, sigma_draws)
    stats_rows[name] = {"n": values.size, "mean": values.mean(), "sd": values.std(ddof=1)}
    # With 3 values or fewer, mu's posterior (a t with 2 df or fewer) has no SD.
    mu_sd = mu_draws.std(ddof=1) if values.size > 3 else np.nan
    posterior_rows.append([name, "Population mean mu", mu_draws.mean(), np.median(mu_draws), mu_sd,
                           np.quantile(mu_draws, low), np.quantile(mu_draws, high),
                           credible if not posterior_rows else np.nan])
    posterior_rows.append(["", "One new value", new.mean(), np.median(new), np.nan,
                           np.quantile(new, low), np.quantile(new, high), np.nan])
    chance_rows.append([name, comparison, np.mean(mu_draws < comparison), np.mean(mu_draws > comparison)])

columns = ["variable", "estimating", "mean", "median", "sd", "lower", "upper", "credible level"]
descriptive = pd.DataFrame(stats_rows)
posterior = pd.DataFrame(posterior_rows, columns=columns)
print("\nDescriptive statistics")
print(descriptive)
print(f"\nPosterior of the population mean and of one new value ({credible:.0%} intervals)")
print(posterior.to_string(index=False))
chance = pd.DataFrame(chance_rows, columns=["variable", "comparison", "P(mu < c)", "P(mu > c)"])
print("\nProbability of being below or above the comparison value")
print(chance.to_string(index=False))

## What the result says

The 20 orders took 68.4 hours on average. The posterior is what we believe about the courier's true average after seeing the data. Its 95% credible interval, the range that holds the true average with 95% probability, runs from 63.7 to 73.1 hours. The probability that the true average is under 72 hours is 0.936, about 94%.

One customer's parcel is a different question. The 95% interval for one new value runs from 46.7 to 90.1 hours. It is much wider, because single parcels vary far more than an average does.

So the courier very likely keeps its promise on average. But many single orders will still take longer than 72 hours, and single late parcels are what customers complain about.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.